In [1]:
import pandas as pd
import numpy as np

# Display settings (important for debugging)
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

In [2]:
panel = pd.read_csv('../data/raw/country_year_final_panel_full.csv')
cases = pd.read_csv('../data/processed/cases_v2.csv')

In [3]:
panel.shape, cases.shape

((8580, 189), (1077, 266))

In [11]:
panel[['COUNTRY', 'year']].head()

,COUNTRY,year
0,ASEAN,1894
1,ASEAN,1895
2,ASEAN,1896
3,ASEAN,1897
4,ASEAN,1898


In [12]:
cases[['country', 'year']].head()

,country,year
0,Angola,2015
1,Angola,2016
2,Australia,1994
3,Australia,1998
4,Australia,2004


In [15]:
len(set(panel['COUNTRY'])), len(set(cases['country']))

(65, 48)

In [16]:
panel['COUNTRY'] = panel['COUNTRY'].str.strip().str.lower()
cases['country'] = cases['country'].str.strip().str.lower()

In [17]:
panel[['COUNTRY', 'year']].duplicated().sum()

np.int64(0)

In [18]:
panel.sort_values(['COUNTRY', 'year']).groupby('COUNTRY')['year'].diff().value_counts().head(10)

year
1.0    8515
Name: count, dtype: int64

In [19]:
lag_block = panel.loc[:, 'de_jure':'REG_RULES_CRISIS']

In [20]:
panel = panel.sort_values(['COUNTRY', 'year'])

In [21]:
lag1 = lag_block.groupby(panel['COUNTRY']).shift(1)
lag2 = lag_block.groupby(panel['COUNTRY']).shift(2)

In [22]:
lag1 = lag1.add_suffix('_lag1')
lag2 = lag2.add_suffix('_lag2')

In [23]:
panel_with_lags = pd.concat([panel, lag1, lag2], axis=1)

In [24]:
panel_with_lags[panel_with_lags['COUNTRY'] == 'angola'][[
    'year',
    'de_jure',
    'de_jure_lag1',
    'de_jure_lag2'
]].head(10)

,year,de_jure,de_jure_lag1,de_jure_lag2
396,1894,0.0,NaN,NaN
397,1895,0.0,0.0,NaN
398,1896,0.0,0.0,0.0
399,1897,0.0,0.0,0.0
400,1898,0.0,0.0,0.0
401,1899,0.0,0.0,0.0
402,1900,0.0,0.0,0.0
403,1901,0.0,0.0,0.0
404,1902,0.0,0.0,0.0
405,1903,0.0,0.0,0.0


In [25]:
cases_with_lags = cases.merge(
    panel_with_lags,
    left_on=['country', 'year'],
    right_on=['COUNTRY', 'year'],
    how='left'
)

In [26]:
cases_with_lags = cases_with_lags.drop(columns=['COUNTRY'])

In [32]:
lag_cols = [col for col in panel_with_lags.columns if 'lag1' in col or 'lag2' in col]

panel_lags_only = panel_with_lags[['COUNTRY', 'year'] + lag_cols]

In [33]:
cases_with_lags = cases.merge(
    panel_lags_only,
    left_on=['country', 'year'],
    right_on=['COUNTRY', 'year'],
    how='left'
)

In [34]:
cases_with_lags = cases_with_lags.drop(columns=['COUNTRY'])

In [35]:
cases_with_lags[['de_jure_lag1', 'de_jure_lag2']].isna().mean()

de_jure_lag1    0.0
de_jure_lag2    0.0
dtype: float64

In [36]:
cases_with_lags.shape

(1077, 636)

In [37]:
cases_with_lags[['year', 'de_jure_lag1', 'de_jure_lag2']].sort_values('year').head(20)

,year,de_jure_lag1,de_jure_lag2
324,1927,0.0,0.0
325,1936,0.0,0.0
326,1940,0.0,0.0
756,1940,0.0,0.0
327,1942,0.0,0.0
328,1951,0.0,0.0
329,1952,0.0,0.0
330,1963,0.0,0.0
758,1964,0.0,0.0
331,1964,0.0,0.0


In [38]:
cases_with_lags['year'].min(), cases_with_lags['year'].max()

(np.int64(1927), np.int64(2025))

In [39]:
cases_with_lags[cases_with_lags['year'] == cases_with_lags['year'].min()][
    ['country', 'year', 'de_jure_lag1', 'de_jure_lag2']
]

,country,year,de_jure_lag1,de_jure_lag2
324,united states,1927,0.0,0.0


In [40]:
panel_with_lags[panel_with_lags['year'] < 1900][[
    'COUNTRY', 'year', 'de_jure_lag1', 'de_jure_lag2'
]].head(10)

,COUNTRY,year,de_jure_lag1,de_jure_lag2
132,african union,1894,NaN,NaN
133,african union,1895,0.0,NaN
134,african union,1896,0.0,0.0
135,african union,1897,0.0,0.0
136,african union,1898,0.0,0.0
137,african union,1899,0.0,0.0
264,algeria,1894,NaN,NaN
265,algeria,1895,0.0,NaN
266,algeria,1896,0.0,0.0
267,algeria,1897,0.0,0.0


In [41]:
cases_with_lags.to_csv('../data/processed/cases_v2_dejure_lags.csv', index=False)